In [3]:
!pip -q uninstall -y numpy pandas pyarrow datasets transformers accelerate evaluate sentencepiece protobuf
!pip -q install -U pip setuptools wheel

!pip -q install --no-cache-dir "numpy==1.26.4" "pandas==2.2.2" "pyarrow==16.1.0"

!pip -q install --no-cache-dir "protobuf<6"

!pip -q install --no-cache-dir \
  "transformers==4.44.2" "accelerate==0.33.0" "datasets==2.21.0" "evaluate==0.4.3" \
  "sentencepiece" "scikit-learn"

print("Installed. NOW: Runtime → Restart runtime")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cufflinks 0.17.3 requires numpy>=1.9.2, which is not installed.
cufflinks 0.17.3 requires pandas>=0.19.2, which is not installed.
pandas-gbq 0.30.0 requires numpy>=1.18.1, which is not installed.
pandas-gbq 0.30.0 requires pandas>=1.1.4, which is not installed.
pandas-gbq 0.30.0 requires pyarrow>=4.0.0, which is not installed.
bigframes 2.38.0 requires numpy>=1.24.0, which is not installed.
bigframes 2.38.0 requires pandas>=1.5.3, which is not installed.
bigframes 2.38.0 requires pyarrow>=15.0.2, which is not installed.
spacy 3.8.11 requires numpy>=1.19.0; python_version >= "3.9", which is not installed.
osqp 1.1.1 requires numpy>=1.7, which is not installed.
arv

In [1]:
import os, json, random
import numpy as np
import pandas as pd
import torch

import transformers, accelerate, datasets, evaluate, google.protobuf
from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("evaluate:", evaluate.__version__)
print("protobuf:", google.protobuf.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


torch: 2.10.0+cu128
transformers: 4.44.2
datasets: 2.21.0
accelerate: 0.33.0
evaluate: 0.4.3
protobuf: 5.29.6
cuda available: True
gpu: NVIDIA L4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
GOEMO_PATH = "/content/drive/MyDrive/IRP_ChatBot/dataset/goemotions/go_emotions_dataset.csv"
OUT_DIR    = "/content/drive/MyDrive/IRP_ChatBot/models/goemotions_7class"

os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME = "distilroberta-base"
MAX_LEN    = 128
SEED       = 42

# Trainer hyperparams
LR         = 2e-5
BATCH_TR   = 32
BATCH_EV   = 64
EPOCHS     = 8
WARMUP     = 0.06
WEIGHT_DECAY = 0.01
PATIENCE   = 2

print("Dataset:", GOEMO_PATH)
print("Output:", OUT_DIR)


Dataset: /content/drive/MyDrive/IRP_ChatBot/dataset/goemotions/go_emotions_dataset.csv
Output: /content/drive/MyDrive/IRP_ChatBot/models/goemotions_7class


In [7]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [8]:
TARGET_EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
label2id = {e:i for i,e in enumerate(TARGET_EMOTIONS)}
id2label = {i:e for e,i in label2id.items()}

df = pd.read_csv(GOEMO_PATH)
print("Original shape:", df.shape)

# keep only single-label rows among 7 emotions
df = df[df[TARGET_EMOTIONS].sum(axis=1) == 1].copy()
print("Filtered single-label shape:", df.shape)

df["label"] = df[TARGET_EMOTIONS].idxmax(axis=1)
df["label_id"] = df["label"].map(label2id)

print("Label distribution:\n", df["label"].value_counts())
print("Example rows:\n", df[["text","label","label_id"]].head())


Original shape: (211225, 31)
Filtered single-label shape: (89318, 31)
Label distribution:
 label
neutral     55298
joy          7722
anger        7450
sadness      6224
surprise     5163
disgust      4641
fear         2820
Name: count, dtype: int64
Example rows:
                                                  text    label  label_id
0                                     That game hurt.  sadness         4
2      You do right, if you don't care then fuck 'em!  neutral         6
4   [NAME] was nowhere near them, he was by the Fa...  neutral         6
10  I have, and now that you mention it, I think t...  neutral         6
12                              BUT IT'S HER TURN! /s  neutral         6


In [9]:
train_df, val_df = train_test_split(
    df[["text","label_id"]],
    test_size=0.1,
    random_state=SEED,
    stratify=df["label_id"]
)

print("Train:", len(train_df), " Val:", len(val_df))


Train: 80386  Val: 8932


In [10]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df.reset_index(drop=True))

train_ds, val_ds


(Dataset({
     features: ['text', 'label_id'],
     num_rows: 80386
 }),
 Dataset({
     features: ['text', 'label_id'],
     num_rows: 8932
 }))

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tokenize, batched=True, remove_columns=["text"])

train_tok = train_tok.rename_column("label_id", "labels")
val_tok   = val_tok.rename_column("label_id", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/80386 [00:00<?, ? examples/s]

Map:   0%|          | 0/8932 [00:00<?, ? examples/s]

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "macro_f1": f1}


In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(TARGET_EMOTIONS),
    id2label=id2label,
    label2id=label2id
)


model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
args = TrainingArguments(
    output_dir=OUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,

    learning_rate=LR,
    per_device_train_batch_size=BATCH_TR,
    per_device_eval_batch_size=BATCH_EV,
    num_train_epochs=EPOCHS,
    warmup_ratio=WARMUP,
    weight_decay=WEIGHT_DECAY,

    fp16=True,
    gradient_checkpointing=False,
    dataloader_num_workers=2,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    seed=SEED,
    report_to="none"
)


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.819100,0.789205,0.733654,0.572837
2,0.737100,0.799987,0.729512,0.587446
3,0.711900,0.776315,0.737013,0.590676
4,0.636900,0.809834,0.731191,0.589073
5,0.595500,0.836327,0.721787,0.586857


TrainOutput(global_step=12565, training_loss=0.73150638712387, metrics={'train_runtime': 827.9107, 'train_samples_per_second': 776.76, 'train_steps_per_second': 24.283, 'total_flos': 3799527044386500.0, 'train_loss': 0.73150638712387, 'epoch': 5.0})

In [ ]:
pred = trainer.predict(val_tok)
logits = pred.predictions
labels = pred.label_ids
preds  = np.argmax(logits, axis=1)

print("Val Accuracy:", accuracy_score(labels, preds))
print("Val Macro-F1:", f1_score(labels, preds, average="macro"))
print("\nClassification report:\n")
print(classification_report(labels, preds, target_names=TARGET_EMOTIONS, digits=4))


Val Accuracy: 0.737012987012987
Val Macro-F1: 0.590676076140017

Classification report:

              precision    recall  f1-score   support

       anger     0.5300    0.5221    0.5260       745
     disgust     0.4899    0.3664    0.4192       464
        fear     0.5343    0.5248    0.5295       282
         joy     0.6994    0.6360    0.6662       772
     sadness     0.6463    0.5104    0.5704       623
    surprise     0.5956    0.5795    0.5874       516
     neutral     0.8112    0.8622    0.8359      5530

    accuracy                         0.7370      8932
   macro avg     0.6152    0.5716    0.5907      8932
weighted avg     0.7287    0.7370    0.7312      8932



In [ ]:
best_dir = os.path.join(OUT_DIR, "best_model")
os.makedirs(best_dir, exist_ok=True)

trainer.save_model(best_dir)
tokenizer.save_pretrained(best_dir)

with open(os.path.join(best_dir, "label_map.json"), "w") as f:
    json.dump(label2id, f, indent=2)

print("Saved best model to:", best_dir)


Saved best model to: /content/drive/MyDrive/IRP_ChatBot/models/goemotions_7class/best_model


In [5]:
best_dir = "/content/drive/MyDrive/IRP_ChatBot/models/goemotions_7class/best_model"

In [28]:
import torch

def predict_emotion(texts, model_dir=best_dir):
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir)
    mdl.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mdl.to(device)

    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN)
    enc = {k:v.to(device) for k,v in enc.items()}

    with torch.no_grad():
        logits = mdl(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        pred_ids = probs.argmax(axis=1)

    return [(texts[i], id2label[int(pred_ids[i])], float(probs[i][pred_ids[i]])) for i in range(len(texts))]

tests = [
    # Anger
    "This situation is making me so angry.",
    "I'm frustrated and losing my patience.",
    "I am really upset about how things turned out.",

    # Disgust
    "That was really gross and unpleasant.",
    "This smells awful and makes me feel sick.",
    "I feel uncomfortable just thinking about it.",

    # Fear
    "I'm really worried about what might happen.",
    "This makes me feel anxious and scared.",
    "I can't stop thinking something bad will happen.",

    # Joy
    "I feel so happy and excited today!",
    "This made my day so much better.",
    "Everything is going perfectly for me.",

    # Sadness
    "I feel really down and alone.",
    "I just feel so empty inside.",
    "I wish things were different.",

    # Surprise
    "Wow, that really caught me off guard!",
    "I didn't expect this at all.",
    "That was such a surprise.",

    # Neutral
    "I am just doing my usual tasks.",
    "Today feels like a normal day.",
    "I have nothing special planned."
]

predict_emotion(tests)


[('This situation is making me so angry.', 'anger', 0.8996168971061707),
 ("I'm frustrated and losing my patience.", 'anger', 0.7168178558349609),
 ('I am really upset about how things turned out.',
  'sadness',
  0.904153048992157),
 ('That was really gross and unpleasant.', 'disgust', 0.9000135660171509),
 ('This smells awful and makes me feel sick.', 'disgust', 0.6351487040519714),
 ('I feel uncomfortable just thinking about it.',
  'sadness',
  0.4380318224430084),
 ("I'm really worried about what might happen.", 'fear', 0.8299685120582581),
 ('This makes me feel anxious and scared.', 'fear', 0.9373054504394531),
 ("I can't stop thinking something bad will happen.",
  'sadness',
  0.33527711033821106),
 ('I feel so happy and excited today!', 'joy', 0.9781215190887451),
 ('This made my day so much better.', 'joy', 0.9620392322540283),
 ('Everything is going perfectly for me.', 'neutral', 0.8174368143081665),
 ('I feel really down and alone.', 'sadness', 0.8899197578430176),
 ('I jus

In [29]:
import pandas as pd

predictions = predict_emotion(tests)

df = pd.DataFrame(predictions, columns=[
    "Input Text",
    "Predicted Emotion",
    "Confidence Score"
])

df

,Input Text,Predicted Emotion,Confidence Score
0,This situation is making me so angry.,anger,0.899617
1,I'm frustrated and losing my patience.,anger,0.716818
2,I am really upset about how things turned out.,sadness,0.904153
3,That was really gross and unpleasant.,disgust,0.900014
4,This smells awful and makes me feel sick.,disgust,0.635149
5,I feel uncomfortable just thinking about it.,sadness,0.438032
6,I'm really worried about what might happen.,fear,0.829969
7,This makes me feel anxious and scared.,fear,0.937305
8,I can't stop thinking something bad will happen.,sadness,0.335277
9,I feel so happy and excited today!,joy,0.978122


In [30]:
df["Confidence Score"] = df["Confidence Score"].round(3)

In [31]:
expected_labels = [
    "anger","anger","anger",
    "disgust","disgust","disgust",
    "fear","fear","fear",
    "joy","joy","joy",
    "sadness","sadness","sadness",
    "surprise","surprise","surprise",
    "neutral","neutral","neutral",
]

df["Expected Emotion"] = expected_labels

In [32]:
df["Correct"] = df["Predicted Emotion"] == df["Expected Emotion"]

In [34]:
import pandas as pd
from IPython.display import display

predictions = predict_emotion(tests)

df = pd.DataFrame(predictions, columns=[
    "Input Text", "Predicted Emotion", "Confidence Score"
])

df["Expected Emotion"] = expected_labels
df["Correct"] = df["Predicted Emotion"] == df["Expected Emotion"]
df["Confidence Score"] = df["Confidence Score"].round(3)
df.insert(0, "Test ID", range(1, len(df) + 1))

display(df)

accuracy = df["Correct"].mean() * 100
print(f"Accuracy on 21-sample qualitative test set: {accuracy:.2f}%")
print("Correct predictions:", df["Correct"].sum())
print("Incorrect predictions:", (~df["Correct"]).sum())

,Test ID,Input Text,Predicted Emotion,Confidence Score,Expected Emotion,Correct
0,1,This situation is making me so angry.,anger,0.900,anger,True
1,2,I'm frustrated and losing my patience.,anger,0.717,anger,True
2,3,I am really upset about how things turned out.,sadness,0.904,anger,False
3,4,That was really gross and unpleasant.,disgust,0.900,disgust,True
4,5,This smells awful and makes me feel sick.,disgust,0.635,disgust,True
5,6,I feel uncomfortable just thinking about it.,sadness,0.438,disgust,False
6,7,I'm really worried about what might happen.,fear,0.830,fear,True
7,8,This makes me feel anxious and scared.,fear,0.937,fear,True
8,9,I can't stop thinking something bad will happen.,sadness,0.335,fear,False
9,10,I feel so happy and excited today!,joy,0.978,joy,True


Accuracy on 21-sample qualitative test set: 76.19%
Correct predictions: 16
Incorrect predictions: 5
